# 02 · Nested spatial cross-validation (regression)

```
for each target, block size (106 m main; 212 and 318 m sensitivity) and model:
    for each outer fold k = 1..5 (V3's blockCV folds):
        test  = development boreholes in fold k
        train = the other development boreholes
        inner CV = the other 4 outer folds (V3's backup inner folds if one is too small)
            Optuna tries 30 hyperparameter sets, each scored by the inner out-of-fold RMSE
        refit the best set on all training boreholes -> predict fold k
```

Models: **RF**, **XGB** (inputs Easting, Northing, mid elevation), **RF_soil**, **XGB_soil** (the same plus soil type, Clay/Sand, as Khamjan et al. 2026) and the **depth-only reference** (training mean per 1 m slice; no tuning).

Soil type is tested honestly: the `_soil` models are trained with the logged soil type, but each outer test fold gets the type **predicted by the classifier** while that fold was out (V3's out-of-fold XGB predictions, same folds). The score with the logged type is kept too, to show how much classification errors cost. Su is clay-only, so it has no `_soil` models.

**Step 1, pilot (a few minutes):** with `PILOT = True` only the first target, XGB, main block, fold 1 runs. Check that it finishes and note its time.
**Step 2, full run:** set `PILOT = False` and run again. It saves after every fold; if the PC stops, run again and it continues. Use `RUN_TARGETS` to run one target at a time (e.g. `["SPT_N"]` first).

**Run time (estimate, not yet measured):** a few minutes per outer fold for RF, less for XGB, so roughly 2-3 hours per target for all three block sizes (5 models). The pilot gives the real figure on your PC.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
import joblib
from tqdm import tqdm

PILOT = True           # True: first target, XGB, main block, fold 1 only.  False: everything.
RUN_TARGETS = None     # e.g. ["SPT_N"] to run one target first

runs = H.run_list(S)
outer = S["quick"]["outer_folds"] if S["run"]["quick"] else list(range(1, S["cv"]["k"] + 1))
main_block = H.block_sizes(S)[0]
if PILOT:
    runs, outer = [r for r in runs if r["model"] == "XGB" and r["block_m"] == main_block][:1], [1]
elif RUN_TARGETS:
    runs = [r for r in runs if r["target"] in RUN_TARGETS]
PRIV = H.out_dir(S, "02_spatial_cv", private=True)
(PRIV / "checkpoints").mkdir(exist_ok=True)
dev = {t: pd.read_csv(H.processed_dir(S) / f"samples_{t}.csv").query("set == 'development'") for t in {r["target"] for r in runs}}
display(pd.DataFrame(runs))

## Run (or continue)
One bar per run; it moves one step per Optuna trial (one step per fold for the depth reference). Next to it: the mean RMSE and R² of the finished outer folds.

In [ ]:
for r in runs:
    per_fold = max(H.n_trials(r["model"], S), 1)
    ck = lambda k: PRIV / "checkpoints" / f"{r['tag']}_fold{k}.joblib"
    done = {k: H.scores(*joblib.load(ck(k))["predictions"][["y", "y_pred"]].values.T) for k in outer if ck(k).exists()}
    bar = tqdm(total=len(outer) * per_fold, initial=len(done) * per_fold, desc=f"{r['tag']:<26}",
               bar_format="{desc} {percentage:3.0f}%|{bar:25}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]{postfix}")

    def show():
        if done:
            bar.set_postfix_str(f"folds {len(done)}/{len(outer)} | mean RMSE {np.nanmean([v['rmse'] for v in done.values()]):.2f}"
                                f" R2 {np.nanmean([v['r2'] for v in done.values()]):.3f}")
    show()
    for k in outer:
        if k in done:
            continue
        res = H.run_fold(r, dev[r["target"]], k, S, on_trial=lambda: bar.update(1))
        if r["model"] == "Depth":
            bar.update(1)
        res.pop("model")
        joblib.dump(res, ck(k), compress=3)
        done[k] = H.scores(res["predictions"].y, res["predictions"].y_pred)
        show()
    bar.close()
print(f"Done ({(time.time() - T0) / 3600:.2f} h in this session).")

## Pilot time (only with PILOT = True)

In [ ]:
if PILOT:
    res = joblib.load(PRIV / "checkpoints" / f"{runs[0]['tag']}_fold1.joblib")
    print(f"Pilot {runs[0]['tag']} fold 1: {res['info']['test_samples']} test samples, "
          f"{res['info']['seconds'] / 60:.1f} min.  Full run is about {len(H.run_list(S))} runs x 5 folds;"
          " RF folds take longer than XGB, the depth reference takes seconds.")
    print("If this worked: set PILOT = False and run the notebook again.")

## Collect the results (after the full run)

In [ ]:
if not PILOT:
    preds, infos = [], []
    for f in sorted((PRIV / "checkpoints").glob("*.joblib")):
        res = joblib.load(f)
        preds.append(res["predictions"]); infos.append(res["info"])
    preds = pd.concat(preds, ignore_index=True)
    preds.to_csv(PRIV / "outer_predictions.csv", index=False)
    info = pd.DataFrame(infos)
    info.drop(columns=["params"]).to_csv(H.out_dir(S, "02_spatial_cv") / "run_info.csv", index=False)
    display(info.groupby("tag").agg(folds=("outer_fold", "count"), hours=("seconds", lambda s: s.sum() / 3600),
                                    inner_folds=("inner_folds", lambda s: ", ".join(sorted(set(s))))).round(2))
    print(f"{len(preds)} test predictions saved. Next: notebook 03.")